# Multi-Head Latent Attention Practice

**Topic:** Multi-Head Latent Attention &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `x` `(2, 10, 32)`: batch 2, sequence 10, model dim 32, with 4 heads × head_dim 8.
- `W_dkv (32, 6)`: down-projection to the 6-dim latent (this is what a real model would cache).
- `W_uk, W_uv (6, 32)`: up-projections from latent to per-head keys/values.
- `Wq (32, 32)`, `Wo (32, 32)`: the usual query and output projections.
- Attention is **not** masked.

## Your task

With **NumPy only**, compute MLA:

1. Latent: `c = x @ W_dkv` → `(2, 10, 6)`.
2. Expand: `k = c @ W_uk`, `v = c @ W_uv` → `(2, 10, 32)`, reshaped to 4 heads of dim 8 (queries come from `x @ Wq` the same way).
3. Standard per-head softmax attention (scale by √8), merge heads, apply `Wo`, and set `mla_out` to the `(2, 10, 32)` result.

## Evaluation

Max absolute difference from the reference must be below 1e-6.


In [ ]:
import numpy as np

rng = np.random.default_rng(441)
B, N, D = 2, 10, 32
N_HEADS, HEAD_DIM, LATENT = 4, 8, 6

x = rng.normal(0, 1.0, (B, N, D))
Wq = rng.normal(0, 0.3, (D, N_HEADS * HEAD_DIM))
W_dkv = rng.normal(0, 0.3, (D, LATENT))
W_uk = rng.normal(0, 0.3, (LATENT, N_HEADS * HEAD_DIM))
W_uv = rng.normal(0, 0.3, (LATENT, N_HEADS * HEAD_DIM))
Wo = rng.normal(0, 0.3, (N_HEADS * HEAD_DIM, D))
print(f"x {x.shape}, cached floats per token: {LATENT} (MLA) vs {2 * D} (vanilla MHA)")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: compare against an independent reference implementation
# =====================================================================
import numpy as np

if "mla_out" not in dir():
    raise NameError("Define `mla_out` in the solution cell before running this cell.")

def _softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def _heads(t):
    return t.reshape(B, N, N_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)

_c = x @ W_dkv
_q, _k, _v = _heads(x @ Wq), _heads(_c @ W_uk), _heads(_c @ W_uv)
_w = _softmax(_q @ _k.transpose(0, 1, 3, 2) / np.sqrt(HEAD_DIM))
_ref = ((_w @ _v).transpose(0, 2, 1, 3).reshape(B, N, N_HEADS * HEAD_DIM)) @ Wo

_out = np.asarray(mla_out, dtype=float)
if _out.shape != _ref.shape:
    raise ValueError(f"`mla_out` has shape {_out.shape}, expected {_ref.shape}.")
_err = float(np.max(np.abs(_out - _ref)))
print("=" * 52)
print(f"Max absolute error vs reference: {_err:.2e}   (need < 1e-6)")
print("=" * 52)
print("PASSED: latent attention, decompressed correctly!" if _err < 1e-6 else "NOT PASSED yet: check the latent path and head reshapes.")


## Solution

Compress once (`c = x @ W_dkv`, the 6-dim latent a real model would cache), expand keys and values from the latent, and run standard 4-head attention. The only new step versus MHA is the down/up projection pair.

```python
import numpy as np

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def heads(t):
    return t.reshape(B, N, N_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)

c = x @ W_dkv                     # (B, N, 6). This is what gets cached
q = heads(x @ Wq)
k = heads(c @ W_uk)
v = heads(c @ W_uv)

w = softmax(q @ k.transpose(0, 1, 3, 2) / np.sqrt(HEAD_DIM))
merged = (w @ v).transpose(0, 2, 1, 3).reshape(B, N, N_HEADS * HEAD_DIM)
mla_out = merged @ Wo
print(mla_out.shape)
```
